# 🍽️ POC v16 — Photo de plat → 3D avec TRELLIS 2 (ton fork GitHub Omazi)

**Réglage :** `Exécution > Modifier le type d'exécution > A100 GPU` + **RAM élevée**.

- **Modèles** : chargés depuis ta sauvegarde Drive (`trellis2_backup`), sans Hugging Face ni jeton.
  Le jeton `HF_TOKEN` ne sert que si cette sauvegarde est introuvable.
- **Site photo → 3D → AR** : lance les cellules 1, 2 puis **7** et scanne le QR code avec ton téléphone.
- Le code vient de **github.com/Omazi** (TRELLIS.2, CuMesh, FlexGEMM) : les correctifs y sont déjà.

In [ ]:
# CELLULE 1 — GPU + Drive + source des modèles
import os, torch
from google.colab import drive
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'AUCUN'
mem = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
print(f'GPU : {gpu} ({mem:.0f} Go) | torch {torch.__version__}')
assert mem >= 22, "❌ Il faut un A100 : Exécution > Modifier le type d'exécution."

drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/trellis2_backup'
HF_CACHE = f'{BACKUP}/hf_cache'
SMALL = f'{BACKUP}/small_models'          # détourage / détection / segmentation (cellule 6)
if os.path.isdir(SMALL):
    os.environ['U2NET_HOME'] = f'{SMALL}/u2net'
    os.environ['MENU_AR_WEIGHTS'] = f'{SMALL}/ultralytics'
    os.system(f'mkdir -p ~/.cache && cp -rn "{SMALL}/clip" ~/.cache/ 2>/dev/null')
    print('✅ Petits modèles chargés depuis ton Drive')
# Modèles Hugging Face. Une copie complète sur ton Drive est lue directement ; un modèle absent ou
# incomplet est téléchargé sur le disque de la session (fiable, rapide), puis recopié sur ton Drive
# en arrière-plan. (Télécharger directement dans Drive coupait les gros fichiers : c'est ce qui avait
# empêché la texture IA de se charger.)
import shutil, threading
LOCAL = '/content/hf_cache'; os.makedirs(LOCAL, exist_ok=True)
os.environ['HF_HUB_CACHE'] = LOCAL
MODELS = {   # dépôt -> fichiers utiles (None = tout le dépôt)
    'microsoft/TRELLIS.2-4B': None, 'microsoft/TRELLIS-image-large': None,
    'facebook/dinov3-vitl16-pretrain-lvd1689m': None,
    'stabilityai/stable-diffusion-xl-base-1.0': ['model_index.json', 'scheduler/*', 'tokenizer/*', 'tokenizer_2/*',
        'text_encoder/config.json', 'text_encoder/model.fp16.safetensors', 'text_encoder_2/config.json',
        'text_encoder_2/model.fp16.safetensors', 'unet/config.json', 'unet/diffusion_pytorch_model.fp16.safetensors',
        'vae/config.json'],
    'madebyollin/sdxl-vae-fp16-fix': ['config.json', 'diffusion_pytorch_model.safetensors'],
    'huanngzh/mv-adapter': ['mvadapter_ig2mv_sdxl.safetensors'],
}
def _dir(root, repo): return f"{root}/models--{repo.replace('/', '--')}"
def drive_ok(repo):
    d = _dir(HF_CACHE, repo)
    if os.path.exists(f'{d}/.complet'): return True
    if not os.path.isdir(f'{d}/snapshots'): return False
    w = [os.path.join(r, f) for r, _, fs in os.walk(f'{d}/snapshots') for f in fs
         if f.endswith(('.safetensors', '.pt', '.pth', '.bin', '.ckpt'))]
    ok = bool(w) and all(os.path.exists(x) and os.path.getsize(os.path.realpath(x)) > 1_000_000 for x in w)
    if ok: open(f'{d}/.complet', 'w').close()
    return ok
missing = []
for repo in MODELS:
    loc = _dir(LOCAL, repo)
    if os.path.isdir(HF_CACHE) and drive_ok(repo):
        if not os.path.exists(loc): os.symlink(_dir(HF_CACHE, repo), loc)
    elif not os.path.isdir(loc) or os.path.islink(loc):
        missing.append(repo)
if missing:
    print('⏳ À télécharger une fois (puis gardés sur ton Drive) :', ', '.join(missing))
    from huggingface_hub import login, snapshot_download
    try:
        from google.colab import userdata; login(userdata.get('HF_TOKEN')); print('✅ Jeton Hugging Face OK')
    except Exception:
        if any('dinov3' in m for m in missing):
            raise SystemExit('❌ DINOv3 doit être téléchargé et le secret HF_TOKEN est absent.')
    for repo in missing:
        loc = _dir(LOCAL, repo)
        if os.path.islink(loc): os.unlink(loc)
        snapshot_download(repo, allow_patterns=MODELS[repo], cache_dir=LOCAL)
        print('  ✅', repo)
    def _to_drive(repos):     # copie sur ton Drive : refs + fichiers réels (sans liens), marqueur à la fin
        for repo in repos:
            src, dst = _dir(LOCAL, repo), _dir(HF_CACHE, repo)
            tmp = dst + '.copie'
            shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
            shutil.copytree(f'{src}/refs', f'{tmp}/refs')
            shutil.copytree(f'{src}/snapshots', f'{tmp}/snapshots', symlinks=False)
            open(f'{tmp}/.complet', 'w').close()
            shutil.rmtree(dst, ignore_errors=True); os.rename(tmp, dst)
            print(f'💾 {repo} sauvegardé sur ton Drive')
    if os.path.isdir(BACKUP):
        os.makedirs(HF_CACHE, exist_ok=True)
        threading.Thread(target=_to_drive, args=(missing,), daemon=True).start()
        print('💾 Copie vers ton Drive en arrière-plan (garde la session ouverte quelques minutes)')
os.environ['HF_HUB_OFFLINE'] = '1'
print('✅ Modèles prêts (lus sans Hugging Face)')


In [ ]:
# CELLULE 2 — Installation depuis ton GitHub (15-25 min la 1re fois, ~1 min ensuite)
import os, sys, subprocess, glob, re, torch

def sh(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode:
        errs = [l for l in (r.stdout + r.stderr).splitlines() if re.search(r'error|fatal|killed', l, re.I) and 'Werror' not in l]
        print('\n'.join(dict.fromkeys(errs[:60])) or (r.stdout + r.stderr)[-3000:])
        raise SystemExit(f'❌ Échec : {cmd[:120]} — copie-moi les lignes ci-dessus.')
    return r.stdout

sh('pip uninstall -y -q tensorflow tf-keras tensorflow-hub || true')
if not os.path.exists('/content/TRELLIS.2'):
    sh('git clone -q https://github.com/Omazi/TRELLIS.2.git /content/TRELLIS.2')
else:   # toujours la dernière version de ton GitHub (même si un fichier a été modifié ici)
    sh('git -C /content/TRELLIS.2 fetch -q origin main && git -C /content/TRELLIS.2 reset -q --hard origin/main')
CODE_REV = sh('git -C /content/TRELLIS.2 rev-parse --short HEAD').strip()
# Code mis à jour dans une session déjà lancée : on oublie l'ancien code Python chargé
_ours = ('trellis2.menu_ar_pipeline', 'trellis2.utils.dish_segmentation', 'trellis2.utils.photo_projection',
         'trellis2.utils.food_postprocess', 'trellis2.utils.mv_texture', 'mvadapter', 'webapp')
for _m in [m for m in sys.modules if m.startswith(_ours)]:
    del sys.modules[_m]
print('✅ Code', CODE_REV)
sh('pip install -q easydict plyfile zstandard kornia timm imageio imageio-ffmpeg trimesh rembg onnxruntime ninja ultralytics scikit-image '
   'diffusers peft accelerate spandrel==0.4.2 gltflib jaxtyping typeguard omegaconf einops fastapi uvicorn python-multipart qrcode')
sh('pip install -q git+https://github.com/ultralytics/CLIP.git')
sh('pip install -q /content/TRELLIS.2/third_party/utils3d')   # copie dans ton dépôt
# Node.js >= 18 pour l'optimisation mobile du GLB
nv = subprocess.run('node -v', shell=True, text=True, capture_output=True).stdout.strip()
if not nv or int(nv.lstrip('v').split('.')[0]) < 18:
    sh('curl -fsSL https://nodejs.org/dist/v22.11.0/node-v22.11.0-linux-x64.tar.xz | tar -xJ -C /usr/local --strip-components=1')
sh('cd /content/TRELLIS.2/tools/mobile_glb && npm install -q --no-audit --no-fund')
print('✅ Node', subprocess.run('node -v', shell=True, text=True, capture_output=True).stdout.strip())

# Toolkit CUDA de même version majeure que PyTorch
tcuda = torch.version.cuda
def nvcc_ver(home):
    out = subprocess.run(f'{home}/bin/nvcc --version', shell=True, text=True, capture_output=True).stdout
    m = re.search(r'release (\d+\.\d+)', out); return m.group(1) if m else ''
match = [h for h in sorted(glob.glob('/usr/local/cuda-*')) + ['/usr/local/cuda'] if nvcc_ver(h).split('.')[0] == tcuda.split('.')[0]]
if not match:
    maj, mi = tcuda.split('.')
    subprocess.run(f'apt-get -qq update && apt-get -qq install -y cuda-toolkit-{maj}-{mi}', shell=True)
    match = [h for h in sorted(glob.glob('/usr/local/cuda-*')) if nvcc_ver(h).split('.')[0] == maj]
    if not match: raise SystemExit(f'❌ Toolkit CUDA {tcuda} introuvable.')
os.environ['CUDA_HOME'] = match[-1]
os.environ['PATH'] = f'{match[-1]}/bin:' + os.environ['PATH']
cap = torch.cuda.get_device_capability()
os.environ['TORCH_CUDA_ARCH_LIST'] = f'{cap[0]}.{cap[1]}'
os.environ['MAX_JOBS'] = str(os.cpu_count())

# Composants GPU : wheels sauvegardés sur Drive, sinon compilation
tag = f'torch{torch.__version__.replace("+","_")}_py{sys.version_info[0]}{sys.version_info[1]}_sm{cap[0]}{cap[1]}'
cache = f'/content/drive/MyDrive/trellis2_wheels/{tag}' if os.path.ismount('/content/drive') else f'/content/wheels/{tag}'
os.makedirs(cache, exist_ok=True)
exts = {
    'nvdiffrast': 'cp -r /content/TRELLIS.2/third_party/nvdiffrast /tmp/ext/nvdiffrast',   # copie dans ton dépôt
    'cumesh':     'git clone -q https://github.com/Omazi/CuMesh.git /tmp/ext/cumesh',
    'flex_gemm':  'git clone -q --recursive https://github.com/Omazi/FlexGEMM.git /tmp/ext/flex_gemm',
    'o_voxel':    'cp -r /content/TRELLIS.2/o-voxel /tmp/ext/o_voxel',
}
os.makedirs('/tmp/ext', exist_ok=True)
for name, get in exts.items():
    found = glob.glob(f'{cache}/{name.replace("_","*")}*.whl') + glob.glob(f'{cache}/{name}*.whl')
    if found:
        sh(f'pip install -q --no-deps --force-reinstall "{found[0]}"'); print(f'✅ {name} (déjà compilé)'); continue
    print(f'🔨 Compilation de {name}...')
    if not os.path.exists(f'/tmp/ext/{name}'): sh(get)
    sh(f'pip wheel -v --no-build-isolation --no-deps /tmp/ext/{name} -w {cache}')
    whl = sorted(glob.glob(f'{cache}/*.whl'), key=os.path.getmtime)[-1]
    sh(f'pip install -q --no-deps "{whl}"'); print(f'✅ {name} compilé')
print('🎉 Installation terminée')

In [ ]:
# CELLULE 3 — Photo + détourage automatique
# MODE = 'tout'  : (par défaut) garde la nourriture + sauces + assiette/boîte, retire le fond. Aucun clic.
# MODE = 'plat'  : garde uniquement la nourriture.
# MODE = 'clics' : comme 'tout', mais tu cliques toi-même sur ce qu'il faut retirer (si l'auto se trompe).
MODE = 'tout'

import sys, io, base64
sys.path.insert(0, '/content/TRELLIS.2')
from google.colab import files
from IPython.display import display
from PIL import Image
from trellis2.utils.dish_segmentation import AutoDishSegmenter, ClickSegmenter

up = files.upload()
img = Image.open(io.BytesIO(up[list(up.keys())[0]])).convert('RGB'); img.thumbnail((1536, 1536))
OUT = '/content/input_rgba.png'

if MODE in ('tout', 'plat'):
    if 'auto_seg' not in globals():
        auto_seg = AutoDishSegmenter()
    food, hull, found = auto_seg.masks(img)
    cut = auto_seg(img, keep_container=(MODE == 'tout'))
    cut.save(OUT)
    prev = Image.new('RGB', cut.size, (70, 170, 70)); prev.paste(cut, mask=cut.split()[3]); prev.thumbnail((500, 500))
    display(prev)
    print('✅ Détourage automatique. Éléments repérés :', ', '.join(found) or 'nourriture seulement')
    print('   (vert = retiré). Si ce n\'est pas bon : MODE = \'clics\' et relance la cellule.')
else:
    from google.colab import output
    from IPython.display import HTML, JSON
    seg = ClickSegmenter(img)
    seg.rgba().save(OUT)

    def _png():
        b = io.BytesIO(); seg.preview(700).save(b, 'PNG'); return base64.b64encode(b.getvalue()).decode()

    def _reply():
        seg.rgba().save(OUT); return JSON({'img': _png(), 'kept': round(100 * float(seg.mask.mean()))})

    output.register_callback('dish_click', lambda x, y, remove: (seg.click(int(x), int(y), bool(remove)), _reply())[1])
    output.register_callback('dish_undo', lambda: (seg.undo(), _reply())[1])
    W, H = seg.image.size
    html = '''
<div style="font-family:system-ui,sans-serif;max-width:700px">
  <div style="margin:6px 0 10px;display:flex;gap:8px;align-items:center;flex-wrap:wrap">
    <button id="dmode" style="padding:7px 12px;border-radius:8px;border:0;background:#c4501f;color:#fff;font-weight:600;cursor:pointer">Mode : RETIRER</button>
    <button id="dundo" style="padding:7px 12px;border-radius:8px;border:1px solid #bbb;background:#fff;cursor:pointer">Annuler le dernier clic</button>
    <span id="dstat" style="color:#555;font-size:14px">Clique sur la table / le fond à retirer. Le vert = retiré.</span>
  </div>
  <img id="dimg" src="data:image/png;base64,__IMG__" style="max-width:100%;cursor:crosshair;border-radius:8px;display:block">
</div>
<script>
(function () {
  const img = document.getElementById('dimg'), stat = document.getElementById('dstat'), modeBtn = document.getElementById('dmode');
  let remove = true, busy = false;
  modeBtn.onclick = () => {
    remove = !remove;
    modeBtn.textContent = 'Mode : ' + (remove ? 'RETIRER' : 'RAJOUTER');
    modeBtn.style.background = remove ? '#c4501f' : '#2e7d32';
  };
  async function call(fn, args) {
    if (busy) return;
    busy = true; stat.textContent = 'Découpage en cours…';
    try {
      const res = await google.colab.kernel.invokeFunction(fn, args, {});
      const d = res.data['application/json'];
      img.src = 'data:image/png;base64,' + d.img;
      stat.textContent = 'Gardé : ' + d.kept + ' % de la photo. Continue, ou passe à la cellule 4.';
    } catch (e) { stat.textContent = 'Erreur : ' + e; }
    busy = false;
  }
  img.onclick = (e) => {
    const r = img.getBoundingClientRect();
    const x = Math.round((e.clientX - r.left) / r.width * __W__);
    const y = Math.round((e.clientY - r.top) / r.height * __H__);
    call('dish_click', [x, y, remove]);
  };
  document.getElementById('dundo').onclick = () => call('dish_undo', []);
})();
</script>'''
    display(HTML(html.replace('__IMG__', _png()).replace('__W__', str(W)).replace('__H__', str(H))))
    print('👆 Clique sur la photo ci-dessus. Le résultat est enregistré à chaque clic.')

In [ ]:
# CELLULE 4 — Génération 3D haute définition (~1-2 min, + chargement des modèles la 1re fois)
import os, sys, time, torch
sys.path.insert(0, '/content/TRELLIS.2'); os.chdir('/content/TRELLIS.2')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
from PIL import Image
from trellis2.pipelines import Trellis2ImageTo3DPipeline
if 'pipeline' not in globals():
    pipeline = Trellis2ImageTo3DPipeline.from_pretrained('microsoft/TRELLIS.2-4B', load_rembg=False)
    pipeline.cuda()
t = time.time()
mesh = pipeline.run(Image.open('/content/input_rgba.png'), seed=42, pipeline_type='1536_cascade')[0]
print(f'✅ Généré en {time.time()-t:.0f} s')

In [ ]:
# CELLULE 5 — Export GLB : texture IA haute définition (MV-Adapter) + version recalée sur la photo
# 1re fois : télécharge SDXL (~7 Go, sauvegardé sur ton Drive) ; ensuite ~1 min 30 de plus que la v13.
import o_voxel, os, subprocess, sys, numpy as np, trimesh
from PIL import Image
from google.colab import files
sys.path.insert(0, '/content/TRELLIS.2')
from trellis2.utils.food_postprocess import solidify_base, paint_underside
from trellis2.utils.photo_projection import estimate_camera, recolor_from_photo
TAILLE_M = 0.30   # plus grand côté du plat en mètres (taille réelle en AR)

mesh.simplify(16777216)
glb = o_voxel.postprocess.to_glb(
    vertices=mesh.vertices, faces=mesh.faces, attr_volume=mesh.attrs, coords=mesh.coords,
    attr_layout=mesh.layout, voxel_size=mesh.voxel_size, aabb=[[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
    decimation_target=300000, texture_size=4096, remesh=True, remesh_band=1, remesh_project=0, verbose=True,
)
glb.vertices, _ = solidify_base(glb.vertices, glb.faces)   # dessous rabattu à plat sur la table
glb.export('/content/plat_hd.glb')

# Couleurs : recalage de chaque zone sur la photo (corrige la dérive, ex. pâte rendue rose)
scene = trimesh.load('/content/plat_hd.glb'); g = list(scene.geometry.values())[0]
V, F, UV = np.asarray(g.vertices), np.asarray(g.faces), np.asarray(g.visual.uv)
photo = Image.open('/content/input_rgba.png'); P = np.asarray(photo)
tex = np.asarray(g.visual.material.baseColorTexture.convert('RGB'), np.float32) / 255; T = tex.shape[0]
vcol = tex[np.clip(((1 - UV[:, 1]) * (T - 1)).astype(int), 0, T - 1), np.clip((UV[:, 0] * (T - 1)).astype(int), 0, T - 1)]
cam = estimate_camera(V, F, P[..., 3] > 127, verbose=True, vertex_colors=vcol, photo_rgb=P[..., :3] / 255.)
new_tex, st = recolor_from_photo(g.visual.material.baseColorTexture, V, F, UV, photo, cam)
new_tex = paint_underside(new_tex, V, F, UV)                  # fond uni sombre (plus de nourriture « en transparence »)
new_tex.save('/content/texture_photo.png')
def node(*a):
    r = subprocess.run(['node', *a], cwd='/content/TRELLIS.2/tools/mobile_glb', text=True, capture_output=True)
    if r.returncode: raise RuntimeError(r.stderr[-800:])
node('retexture.mjs', '/content/plat_hd.glb', '/content/texture_photo.png', '/content/plat_hd_photo.glb')
print(f"✅ Texture photo : couleurs recalées ({st['visible_vertices']:.0%} du modèle vu par la photo)")

# Texture IA haute définition : MV-Adapter dessine 6 vues guidées par la photo + notre géométrie
import torch
from trellis2.utils.mv_texture import MVTexturer
HD_IA = None
try:
    torch.cuda.empty_cache()
    if 'mv_tex' not in globals():
        mv_tex = MVTexturer()
    os.makedirs('/content/mv', exist_ok=True)
    ia = mv_tex('/content/plat_hd.glb', photo, cam, '/content/mv')
    ia = paint_underside(ia, V, F, UV)                        # même fond uni que la version photo
    ia.save('/content/texture_ia.png')
    node('retexture.mjs', '/content/plat_hd.glb', '/content/texture_ia.png', '/content/plat_hd_ia.glb')
    HD_IA = '/content/plat_hd_ia.glb'
    from IPython.display import display
    v = Image.open('/content/mv/mv_views.png'); v.thumbnail((1800, 300)); display(v)
    print('✅ Texture IA : les 6 vues générées sont affichées ci-dessus')
except Exception as e:
    import traceback; traceback.print_exc(); torch.cuda.empty_cache()
    print('⚠️ Texture IA en échec (copie-moi l’erreur ci-dessus) : on garde la texture photo.')

node('optimize.mjs', HD_IA or '/content/plat_hd_photo.glb', '/content/plat_mobile.glb', str(TAILLE_M), '0.35')
node('optimize.mjs', '/content/plat_hd_photo.glb', '/content/plat_mobile_photo.glb', str(TAILLE_M), '0.35')
for f in ['/content/plat_mobile.glb', '/content/plat_mobile_photo.glb']:
    print(f'✅ {os.path.basename(f)} : {os.path.getsize(f)/1e6:.1f} Mo')
files.download('/content/plat_mobile.glb')          # texture IA
files.download('/content/plat_mobile_photo.glb')    # texture photo (pour comparer)

In [ ]:
# CELLULE 6 — Sauvegarde des modèles sur Drive (une seule fois, ~18 Go, 5-10 min)
import os, subprocess
if os.environ.get('HF_HUB_OFFLINE') == '1':
    print('✅ Déjà chargés depuis ta sauvegarde Drive, rien à faire')
else:
    BACKUP = '/content/drive/MyDrive/trellis2_backup'; os.makedirs(BACKUP, exist_ok=True)
    src = os.path.expanduser('~/.cache/huggingface/hub')
    subprocess.run(f'rsync -a "{src}/" "{BACKUP}/hf_cache/"', shell=True, check=True)
    print(subprocess.run(f'du -sh {BACKUP}/hf_cache', shell=True, capture_output=True, text=True).stdout)
    print('✅ Modèles sauvegardés : les prochaines sessions n’auront plus besoin de Hugging Face')
# petits modèles (détourage BiRefNet, détection YOLOE + CLIP, segmentation SAM 2)
import glob
SMALL = '/content/drive/MyDrive/trellis2_backup/small_models'
os.makedirs(f'{SMALL}/ultralytics', exist_ok=True)
subprocess.run(f'rsync -a ~/.u2net/ "{SMALL}/u2net/" 2>/dev/null; rsync -a ~/.cache/clip/ "{SMALL}/clip/" 2>/dev/null', shell=True)
for f in glob.glob('/content/**/*.pt', recursive=True) + glob.glob('/content/*.pt'):
    if os.path.basename(f).startswith(('yoloe', 'sam', 'mobileclip')) and '/drive/' not in f:
        subprocess.run(['cp', '-n', f, f'{SMALL}/ultralytics/'])
print(subprocess.run(f'du -sh "{SMALL}"/*', shell=True, capture_output=True, text=True).stdout)
subprocess.run(f'cp -n /content/TRELLIS.2/checkpoints/* "{SMALL}/ultralytics/" 2>/dev/null', shell=True)   # Real-ESRGAN + LaMa
print('✅ Petits modèles sauvegardés (lance une génération avant, pour qu’ils soient téléchargés)')

In [ ]:
# CELLULE 7 — Lancer le site : photo -> 3D -> AR depuis ton téléphone
# Lance d'abord les cellules 1 et 2. Les cellules 3 à 5 sont facultatives (test manuel).
# Le site tourne tant que cette session Colab reste ouverte.
import os, sys, re, time, secrets, subprocess, io
sys.path.insert(0, '/content/TRELLIS.2'); os.chdir('/content/TRELLIS.2')
subprocess.run('pip install -q fastapi uvicorn python-multipart qrcode', shell=True)   # (déjà installés en cellule 2)
from IPython.display import display, HTML
from trellis2.menu_ar_pipeline import MenuARPipeline
from webapp.server import create_app, serve

import trellis2.menu_ar_pipeline as _map
# 1. pipeline : reconstruit si le code a changé (sinon le site tournerait avec l'ancienne version)
if globals().get('MENU_REV') != _map.CODE_VERSION:
    import gc, torch
    for _v in ('menu_pipe', 'mv_tex', 'auto_seg'):
        globals().pop(_v, None)
    gc.collect(); torch.cuda.empty_cache()
    menu_pipe = MenuARPipeline(trellis_pipeline=globals().get('pipeline'))   # TRELLIS gardé, le reste rechargé
    pipeline, auto_seg, mv_tex = menu_pipe.trellis, menu_pipe.segmenter, menu_pipe.texturer
    MENU_REV = _map.CODE_VERSION
print('✅ Pipeline version', MENU_REV, '| texture IA :', 'oui' if menu_pipe.texturer else 'NON (voir erreur ci-dessus)')
def site_job(*a, **k):          # le site appelle toujours le pipeline le plus récent
    return globals()['menu_pipe'].run(*a, **k)

# 2. serveur web (plats gardés sur ton Drive s'il est monté)
DATA = '/content/drive/MyDrive/menu_ar_plats' if os.path.ismount('/content/drive') else '/content/menu_ar_plats'
if 'SITE_KEY' in globals() and 'SITE_V15' not in globals():
    raise SystemExit('❌ Un ancien site tourne encore : Exécution > Redémarrer la session, puis cellules 1, 2 et 7.')
if 'SITE_KEY' not in globals():
    SITE_KEY = secrets.token_hex(6); SITE_V15 = True
    serve(create_app(site_job, DATA, SITE_KEY), port=8000)

# 3. tunnel HTTPS public (Cloudflare, sans compte) pour l'ouvrir sur le téléphone
if 'SITE_URL' not in globals():
    if not os.path.exists('/usr/local/bin/cloudflared'):
        subprocess.run('curl -fsSL -o /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
    tunnel = subprocess.Popen(['cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://localhost:8000'],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    SITE_URL = None; t0 = time.time()
    while time.time() - t0 < 60 and not SITE_URL:
        line = tunnel.stdout.readline()
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line or '')
        if m: SITE_URL = m.group(0)
    if not SITE_URL:
        raise SystemExit('❌ Le tunnel ne démarre pas. Relance la cellule.')
    time.sleep(3)   # le temps que l'adresse soit joignable

link = f'{SITE_URL}/?k={SITE_KEY}'
import qrcode
qr = qrcode.make(link); b = io.BytesIO(); qr.save(b, 'PNG')
import base64
display(HTML(f'''<div style="font-family:system-ui;display:flex;gap:20px;align-items:center">
  <img src="data:image/png;base64,{base64.b64encode(b.getvalue()).decode()}" width="190">
  <div><div style="font-size:18px;font-weight:600;margin-bottom:6px">📱 Scanne avec ton téléphone</div>
  <a href="{link}" target="_blank" style="font-size:14px;word-break:break-all">{link}</a>
  <p style="color:#666;font-size:13px;margin-top:8px">Garde ce lien pour toi : la clé à la fin protège ton GPU.<br>
  Les plats générés sont enregistrés dans {DATA}.</p></div></div>'''))